In [86]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from pydantic import BaseModel,Field
from pprint import pprint
import os
import re
import operator


In [87]:
generator_llm = ChatGroq(model="qwen/qwen3-32b", api_key=os.getenv("GROQ_API_KEY"))
evaluator_llm = ChatGroq(model="openai/gpt-oss-120b", api_key=os.getenv("GROQ_API_KEY"))
optimizer_llm = ChatGroq(model="openai/gpt-oss-120b", api_key=os.getenv("GROQ_API_KEY"))


In [88]:
class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(
        ..., description="Final evaluation result."
    )
    feedback: str = Field(
        ..., description="Constructive feedback for the tweet."
    )
    score: int = Field(
        ..., ge=0, le=5, description="Total score from rubric (0 to 5)."
    )

evaluator_structured_llm = evaluator_llm.with_structured_output(TweetEvaluation)

In [89]:
class TweetState(TypedDict):
    topic: str
    tweet: str
    tweet_history: Annotated[list[str], Field(description="History of generated tweets in previous iterations."), operator.add]
    feedback_history: Annotated[list[str], Field(description="History of feedback received in previous iterations."), operator.add]
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str
    iteration: int
    max_iterations: int

In [ ]:
def generate_tweet(state: TweetState) -> dict:
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".
Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
This is version {state['iteration'] + 1}.
""")
    ]
    msg = generator_llm.invoke(messages)
    raw = msg.content

    cleaned = re.sub(r'<think>.*?</think>', '', raw, flags=re.S).strip()

    m = re.search(r'[""](.+?)[""]\s*$', cleaned, flags=re.S)
    if m:
        tweet_text = m.group(1).strip()
    else:
        cleaned = re.sub(r'\(\s*\d+\s*chars?\s*\)\s*$', '', cleaned, flags=re.I).strip()
        lines = [ln.strip() for ln in cleaned.splitlines() if ln.strip()]
        tweet_text = lines[-1] if lines else cleaned

    if re.match(r'^\*?\s*\(\s*\d+\s*chars?\s*\)\s*\*?$', tweet_text, re.I) or len(tweet_text) < 10:
        tweet_text = cleaned  
    return {"tweet": tweet_text, "tweet_history": [tweet_text]}


def evaluate_tweet(state: TweetState) -> dict:
    messages = [
        SystemMessage(
            content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."
        ),
        HumanMessage(content=f"""
Evaluate the following tweet:
Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:
1. Originality Is this fresh, or have you seen it a hundred times before?
2. Humor Did it genuinely make you smile, laugh, or chuckle?
3. Punchiness Is it short, sharp, and scroll-stopping?
4. Virality Potential Would people retweet or share it?
5. Format Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke

### Respond ONLY in structured format:
evaluation: "approved" or "needs_improvement"
feedback: One paragraph explaining the strengths and weaknesses
score: integer 0-5
""")
    ]

    try:
        result = evaluator_structured_llm.invoke(messages)
        raw_score = getattr(result, "score", None)

        try:
            score = int(round(float(raw_score))) if raw_score is not None else 0
        except Exception:
            score = 0

        return {
            "evaluation": result.evaluation,
            "feedback": result.feedback,
            "feedback_history": state["feedback_history"] + [result.feedback],
            "score": score,
        }

    except Exception:
        raw = evaluator_llm.invoke(messages).content

        m_eval = re.search(r'evaluation:\s*["\']?(approved|needs_improvement)["\']?', raw, flags=re.I)
        evaluation = m_eval.group(1) if m_eval else "needs_improvement"

        m_feed = re.search(r'feedback:\s*(.*?)(?:\nscore:|$)', raw, flags=re.S | re.I)
        feedback = m_feed.group(1).strip() if m_feed else raw.strip()

        m_score = re.search(r"score:\s*(\d+)", raw, flags=re.I)
        score = int(m_score.group(1)) if m_score else 0

        return {
            "evaluation": evaluation,
            "feedback": feedback,
            "feedback_history": state["feedback_history"] + [feedback],
            "score": score,
        }


def optimize_tweet(state: TweetState) -> dict:
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
{state['feedback']}
Topic: {state['topic']}
Original Tweet: {state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
Return ONLY the improved tweet text (no extra commentary).
""")
    ]
    response = optimizer_llm.invoke(messages)

    # Clean optimizer output like generate_tweet does
    opt_raw = re.sub(r'<think>.*?</think>', '', response.content, flags=re.S).strip()
    m = re.search(r'["“](.+?)["”]\s*$', opt_raw, flags=re.S)
    if m:
        new_tweet = m.group(1).strip()
    else:
        lines = [ln.strip() for ln in opt_raw.splitlines() if ln.strip()]
        new_tweet = lines[-1] if lines else opt_raw

    iteration = state.get('iteration', 0) + 1

    return {
        "tweet": new_tweet,
        "iteration": iteration,
        "tweet_history": [new_tweet]
    }

def check_optimisation_needed(state: TweetState) -> str:
    if state['evaluation'] == "approved" or state['iteration'] >= state['max_iterations']:
        return "approved"
    return "needs_improvement"

In [91]:
graph = StateGraph(TweetState)

graph.add_node("generate", generate_tweet)
graph.add_node("evaluate", evaluate_tweet)
graph.add_node("optimize", optimize_tweet)
graph.add_edge(START, "generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges("evaluate", check_optimisation_needed, {
    "approved": END,
    "needs_improvement": "optimize"
})
graph.add_edge("optimize", "evaluate")

workflow = graph.compile()

In [98]:
initial_state = {
    "topic": "Odisha",
    "tweet": "",
    "evaluation": "",
    "feedback": "",
    "iteration": 0,
    "max_iterations": 3,
    "tweet_history": [],
    "feedback_history": []
}

result = workflow.invoke(initial_state)
pprint(result)

{'evaluation': 'approved',
 'feedback': 'The tweet is original enough, mixing a quirky food mishap with a '
             'vivid metaphor about Odisha’s spice level that feels fresh. The '
             'humor lands well—the image of a friend drowning kheer in dahi '
             'and the ‘uninvited relative demanding Arctic AC’ is funny and '
             'relatable. While a bit long at 262 characters, it remains '
             'engaging and scroll‑stopping thanks to strong visual language '
             'and emojis. The hashtag and cultural hook give it solid retweet '
             'potential, and it meets all formatting rules.',
 'feedback_history': ['The tweet is original enough, mixing a quirky food '
                      'mishap with a vivid metaphor about Odisha’s spice level '
                      'that feels fresh. The humor lands well—the image of a '
                      'friend drowning kheer in dahi and the ‘uninvited '
                      'relative demanding Arctic AC’

In [97]:
for feedback in result["feedback_history"]:
    print(feedback)

The tweet reads like a self‑referential description rather than an actual punchy observation. While it hints at an ironic contrast, it offers no concrete joke, relatable hook, or meme‑ready phrasing, making it low on humor and virality. Its meta‑format also feels more like a note than a tweet, hurting punchiness and originality. As a result, it’s unlikely to capture scroll‑stoppers or retweets.
The tweet reads like a self‑referential description rather than an actual punchy observation. While it hints at an ironic contrast, it offers no concrete joke, relatable hook, or meme‑ready phrasing, making it low on humor and virality. Its meta‑format also feels more like a note than a tweet, hurting punchiness and originality. As a result, it’s unlikely to capture scroll‑stoppers or retweets.
The tweet blends highbrow science humor with everyday kitchen struggles in a fresh way, earning points for originality. The contrast between quantum theory and a YouTube egg tutorial delivers a light chuc